# Germany 20-zone GenX system: conversion from PyPSA-Eur

Converts a 20-zone PyPSA-Eur network of Germany into the GenX input files of
`example_systems/12_PyPSA_DE_20_zone/`.

**Input:** the CSVs in `DE_20_zone_pypsa/`, exported from PyPSA-Eur's
`base_s_20_elec.nc` with `export_pypsa_eur_de.py`. The network is built with PyPSA-Eur
v2025.04.0 and `config.de20.yaml`; see `README.md`.

**Output:** `DE_20_zone_GenX/` with `Network.csv`, `Demand_data.csv`, `Fuels_data.csv`,
`Generators_variability.csv`, `Thermal.csv`, `Vre.csv` and `Storage.csv`. Copy the first
four into the case's `system/` folder and the last three into `resources/`.

Run the notebook top to bottom with this folder as the working directory, in the Python
environment from `../requirements.txt`. Cost and price assumptions are documented, with
sources, next to the code that uses them.

The case is greenfield: all generation and storage capacity is built by the model.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from itertools import chain
from collections import defaultdict

In [ ]:
import warnings

# Block all warnings
warnings.filterwarnings("ignore")

In [ ]:
n = 20 #number of clusters
# Paths are relative to this notebook's directory (example_systems/pypsa_data_tools/)
folder_path = f"DE_{n}_zone_pypsa/"     # PyPSA-Eur export (input, see README.md)
output_folder = f"DE_{n}_zone_GenX/"    # GenX input files (output)

os.makedirs(output_folder, exist_ok=True)

# All costs data from https://github.com/PyPSA/technology-data/blob/master/outputs/costs_2030.csv

In [ ]:
store_sources = ('battery',)
# store_sources = ('battery', 'PHS')  # uncomment to include PHS

In [ ]:
def annualize(capex, r=0.07, lifetime=40):
    """
    Annualize a capex value
    
    Args:
        capex (float(s)): capital cost (e.g., currency/MW)
        r (float): discount rate (e.g., 5%)
        lifetime (int): lifetime in years (e.g., 40)

    Returns:
        float(s): annualized cost
    """
    crf = r / (1 - (1 + r) ** -lifetime)
    return capex * crf


## Network

In [ ]:
buses = pd.read_csv(folder_path + f"DE_{n}_buses.csv")
lines = pd.read_csv(folder_path + f'DE_{n}_lines.csv')
lines = lines[lines['build_year'] == 0].copy()

In [ ]:
unique_buses = sorted(buses["Bus"].unique(), key=lambda b: int(b.split()[-1]))
bus_to_index = {bus: idx + 1 for idx, bus in enumerate(unique_buses)}
bus_to_genxname = {bus: f"DE{bus_to_index[bus]}" for bus in unique_buses}
bus_to_voltage = dict(zip(buses["Bus"], buses["v_nom"]))

In [ ]:
zones = pd.DataFrame({
    "Bus": unique_buses,
    "Network_zones": [f"z{bus_to_index[bus]}" for bus in unique_buses],
    "Zone_Voltage_Magnitude_Maximum": [1.05 * bus_to_voltage[bus] for bus in unique_buses],
    "Zone_Voltage_Magnitude_Minimum": [0.95 * bus_to_voltage[bus] for bus in unique_buses],
    "Zone_Shunt_Conductance_Siemens": 0,
    "Zone_Shunt_Susceptance_Siemens": 0,
    "Zone_Name": [bus_to_genxname[bus] for bus in unique_buses],
})

In [ ]:
# Direct conversions
lines = lines.rename(columns={
    "v_nom": "Line_Voltage_kV",
    "s_nom": "Line_Max_Flow_MW",
    "s_nom_max": "Line_Max_Reinforcement_MW",
    "num_parallel": "Line_Num_Parallel"
})

# Reindexing
lines["Start_Zone"] = lines["bus0"].map(bus_to_index)
lines["End_Zone"] = lines["bus1"].map(bus_to_index)

# Units and physical conversions
# r, x and b are stored for a single circuit, since GenX scales them by the number of parallel circuits
lines["distance_mile"] = lines["length"] * 0.621371
lines["Line_Resistance_Ohms"] = lines["r"] * lines["Line_Num_Parallel"] 
lines["Line_Reactance_Ohms"] = lines["x"] * lines["Line_Num_Parallel"] 
lines["Line_Charging_Susceptance_Siemens"] = lines["b"] / lines["Line_Num_Parallel"] 

# Economic conversions
# Line reinforcement costs are PyPSA-Eur's line capital costs: technology-data v0.10.1
# (projected 2030, HVAC overhead lines) annualized by PyPSA-Eur at a 7% discount rate.

#Pypsa doc on "captial cost": Fixed period costs of extending s_nom by 1 MVA, including periodized investment costs and periodic fixed O&M costs \
#(e.g. annuitized investment costs). Any length factor must already be included here.
lines["Line_Reinforcement_Cost_per_MWyr"] = lines["capital_cost"]
lines["Line_Max_Reinforcement_MW"] = lines['Line_Max_Flow_MW']

# Additional GenX Line datafields
lines["Network_Lines"] = np.arange(1, len(lines) + 1)
lines["transmission_path_name"] = lines["Start_Zone"].astype(str) + "_to_" + lines["End_Zone"].astype(str)
lines["Single_Line_MW"] = lines["Line_Max_Flow_MW"] / lines["Line_Num_Parallel"]
lines["Line_Loss_Percentage"] = 0.035  # uniform loss for transport/DCOPF, calibrated from ACOPF (see ../calibrate_losses.py)
lines["Angle_Limit_Rad"] = np.pi / 6  # 30° — standard N-1 planning limit for 380 kV
lines["Tap_Ratios"] = 1
lines["Phase_Shift_Degrees"] = 0
lines["CapRes_Excl_1"] = 0
lines["DerateCapRes_1"] = 0.95

In [ ]:
zone_columns = [
    'Network_zones', 'Zone_Voltage_Magnitude_Maximum', 'Zone_Voltage_Magnitude_Minimum',
    'Zone_Shunt_Conductance_Siemens', 'Zone_Shunt_Susceptance_Siemens', 'Zone_Name'
]

line_columns = [
    'Network_Lines', 'Start_Zone', 'End_Zone', 'Line_Max_Flow_MW', 'transmission_path_name',
    'distance_mile', 'Line_Loss_Percentage', 'Single_Line_MW', 'Line_Num_Parallel',
    'Line_Max_Reinforcement_MW', 'Line_Reinforcement_Cost_per_MWyr', 'DerateCapRes_1',
    'CapRes_Excl_1', 'Line_Resistance_Ohms', 'Line_Reactance_Ohms',
    'Line_Charging_Susceptance_Siemens', 'Line_Voltage_kV', 'Angle_Limit_Rad',
    'Tap_Ratios', 'Phase_Shift_Degrees'
]

genx_columns = zone_columns + line_columns

In [ ]:
zones = zones.reset_index(drop=True)
lines = lines.reset_index(drop=True)

n_zones = len(zones)
n_lines = len(lines)
n_overlap = min(n_zones, n_lines)

# Top rows: merge zone + line data row-wise
network_top = pd.concat([
    zones[zone_columns].iloc[:n_overlap].reset_index(drop=True),
    lines[line_columns].iloc[:n_overlap].reset_index(drop=True)
], axis=1)

# Bottom rows: need to pad either zones or lines with NaNs
if n_zones > n_lines:
    network_bottom = pd.concat([
        zones[zone_columns].iloc[n_overlap:].reset_index(drop=True),
        pd.DataFrame(np.nan, index=range(n_zones - n_overlap), columns=line_columns)
    ], axis=1)

elif n_lines > n_zones:
    network_bottom = pd.concat([
        pd.DataFrame(np.nan, index=range(n_lines - n_overlap), columns=zone_columns),
        lines[line_columns].iloc[n_overlap:].reset_index(drop=True)
    ], axis=1)

else:
    network_bottom = pd.DataFrame(columns=genx_columns)

network = pd.concat([network_top, network_bottom], ignore_index=True)[genx_columns]

# cast integer columns
network["Network_Lines"] = network["Network_Lines"].astype("Int64")
network["Start_Zone"] = network["Start_Zone"].astype("Int64")
network["End_Zone"] = network["End_Zone"].astype("Int64")

network.to_csv(output_folder + "Network.csv", index=False)


## Demand

In [ ]:
demand = pd.read_csv(folder_path + f'DE_{n}_loads_t.csv')

In [ ]:
zone_cols = demand.columns[1:]  # skipping snapshot
num_hours = demand.shape[0]
num_zones = len(zone_cols)

# compute reactive demand
np.random.seed(0)
# Reactive demand at a power factor drawn per zone and hour from [0.96, 0.98]. Not used by the
# case, which sets MeanPowerFactor in genx_settings.yml.
power_factors = np.random.uniform(0.96, 0.98, size=(num_hours, num_zones))
# To use a power factor of 1.0:
# power_factors = np.ones((num_hours, num_zones))
active = demand[zone_cols].values
reactive = active * np.tan(np.arccos(power_factors)) 

bus_to_zone = {col: f"z{bus_to_index[col]}" for col in zone_cols}

active_cols = [f"Demand_MW_{bus_to_zone[col]}" for col in zone_cols]
reactive_cols = [f"Reactive_Demand_MVAR_{bus_to_zone[col]}" for col in zone_cols]

active_df = pd.DataFrame(active, columns=active_cols)
reactive_df = pd.DataFrame(reactive, columns=reactive_cols)
time_index = pd.Series(np.arange(1, num_hours + 1), name="Time_Index")

# Step 3: Combine demand data
demand = pd.concat([time_index, active_df, reactive_df], axis=1)


In [ ]:
demand_additional = pd.DataFrame([], columns = ['Voll',
 'Demand_Segment',
 'Cost_of_Demand_Curtailment_per_MW',
 'Max_Demand_Curtailment',
 '$/MWh',
 'Rep_Periods',
 'Timesteps_per_Rep_Period',
 'Sub_Weights'])

# Default values from the GenX example systems
demand_additional.loc[0, 'Voll'] = 16730 #VOLL of Germany in 2023 = 23.2 EUR/KWh; source: Tabelle 9, https://www.bundesnetzagentur.de/DE/Beschlusskammern/BK08/BK8_05_EOG/57_QElement/Download/20251210_BerichtQ26.pdf?__blob=publicationFile&v=2
demand_additional.loc[0, 'Demand_Segment'] = 1
demand_additional.loc[0, 'Cost_of_Demand_Curtailment_per_MW'] = 1
demand_additional.loc[0, 'Max_Demand_Curtailment'] = 1
demand_additional.loc[0, '$/MWh'] = 2000
demand_additional.loc[0, 'Rep_Periods'] = 1
demand_additional.loc[0, 'Timesteps_per_Rep_Period'] = len(demand)
demand_additional.loc[0, 'Sub_Weights'] = len(demand)

In [ ]:
demand = pd.concat([demand_additional, demand], axis = 1)

In [ ]:
demand.to_csv(output_folder + 'Demand_data.csv', index = False)

## Generators

In [ ]:
generator_list = pd.read_csv(folder_path + f'DE_{n}_generators.csv')
generators = generator_list['Generator']

gen_variability = pd.DataFrame([], columns = generators)

In [ ]:
gen_var_res = pd.read_csv(folder_path + f'DE_{n}_generators_t.csv')
gen_var_res.drop(columns = 'snapshot', inplace = True)

In [ ]:
gen_variability = pd.concat([gen_variability, gen_var_res])

In [ ]:
#generator_types
set([s.split()[-1] for s in generators])


In [ ]:
suffixes = ('CCGT', 'OCGT', 'biomass', 'geothermal')
# suffixes = ('CCGT', 'OCGT', 'coal', 'lignite', 'biomass', 'geothermal')  # uncomment to include coal/lignite

thermals = gen_variability.columns[gen_variability.columns.str.endswith(suffixes)]
gen_variability.loc[:, thermals] = 1

In [ ]:
battery = pd.read_csv(folder_path + f'DE_{n}_storage.csv')
batteries = battery[battery['StorageUnit'].str.endswith(store_sources)]['StorageUnit'].to_list()
gen_variability.loc[:, batteries] = 1

In [ ]:
tokens = gen_variability.columns.str.split(r'\s+')
gen_variability.columns = [f"DE{int(t[1]) + 1}_{t[-1]}" for t in tokens]

In [ ]:
gen_variability.reset_index(drop=True, inplace=True)
gen_variability.index = gen_variability.index + 1
gen_variability.index.name = 'Time_Index'

In [ ]:
gen_var_cols = gen_variability.columns

zone_dict = defaultdict(list)
for col in gen_var_cols:
    if '_' in col:
        zone = int(col[2:].split('_')[0])  # Extract number after 'DE'
        zone_dict[zone].append(col)

gv_ordered_cols = list(chain.from_iterable([sorted(zone_dict[z]) for z in sorted(zone_dict)]))

gen_variability = gen_variability[gv_ordered_cols]

In [ ]:
gen_variability.to_csv(output_folder + 'Generators_variability.csv')

## Thermal

In [ ]:
thermal_x = pd.DataFrame([], columns = [
'Resource',
'Zone',
'Model',
'New_Build',
'Can_Retire',
'Existing_Cap_MW',
'Max_Cap_MW',
'Min_Cap_MW',
'Inv_Cost_per_MWyr',
'Fixed_OM_Cost_per_MWyr',
'Var_OM_Cost_per_MWh',
'Heat_Rate_MMBTU_per_MWh',
'Fuel',
'Cap_Size',
'Start_Cost_per_MW',
'Start_Fuel_MMBTU_per_MW',
'Up_Time',
'Down_Time',
'Ramp_Up_Percentage',
'Ramp_Dn_Percentage',
'Min_Power',
'Reg_Max',
'Rsv_Max',
'Reg_Cost',
'Rsv_Cost',
'Biomass',
'region',
'cluster'])

In [ ]:
pyp_gen_therm_dict = {
 'Generator' :  'Resource',
 'bus' : 'Zone',
#  'p_nom' : 'Existing_Cap_MW', # not used: greenfield
 'p_nom_max' : 'Max_Cap_MW',
#  'p_nom_min' : 'Min_Cap_MW', # not used: greenfield
 'capital_cost' : 'Inv_Cost_raw',
 'marginal_cost' : 'Var_OM_Cost_per_MWh',
 'carrier' : 'Fuel',
 'start_up_cost':'Start_Cost_per_MW',
 'min_up_time' : 'Up_Time',
 'min_down_time' : 'Down_Time',
  'ramp_limit_up' : 'Ramp_Up_Percentage',
 'ramp_limit_down' : 'Ramp_Dn_Percentage'
#  'Min_Power' # not set
#  'Reg_Max', # not set
#  'Rsv_Max',# not set
#  'Reg_Cost',# not set
#  'Rsv_Cost',# not set
}

In [ ]:
thermal = generator_list[generator_list['Generator'].str.endswith(suffixes)]
thermal = thermal[list(pyp_gen_therm_dict.keys()) + ['efficiency']]

In [ ]:
thermal = thermal.rename(columns = pyp_gen_therm_dict)

In [ ]:
fuel_type_map = {'CCGT': 'Gas', 'OCGT': 'Gas', 'biomass': 'Biomass'}

tokens = thermal['Resource'].str.split(r'\s+')
thermal['Zone'] = tokens.str[1].astype(int) + 1  # reindex from 1
thermal['Technology'] = tokens.str[-1]
thermal['Resource'] = 'DE' + thermal['Zone'].astype(str) + '_' + thermal['Technology']
thermal['Fuel'] = thermal['Technology'].map(fuel_type_map)
thermal['region'] = 'DE' + thermal['Zone'].astype(str)

In [ ]:
#dictionary of {'technology type' : [Investment (EUR/MW), lifetime (years), FOM (%/year), VOM (EUR/MWh)]}
#source of data: costs_2030.csv from pypsa

#to convert to 2023 currency year, HICP annual data from Eurostat were taken for EU-27:
#https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_aind/default/table?lang=en
#HICP_2015 = 100, HICP 2023 = 126.38

costs_dict = {
    'CCGT': [878.3 * 126.38/100 * 1000, 25, 3.3494/100, 4.4445 * 126.38/100 ], #OG in 2015 currency year, converted to 2023 currency using HICP
    'OCGT': [460.5804 * 126.38/100 * 1000,  25, 1.7795/100, 4.762 * 126.38/100], #OG in 2015 currency year, converted to 2023 currency using HICP
    'coal': [3827.1629 * 1000, 40, 1.31/100, 3.2612], #in 2023 currency year
    'lignite': [3827.1629 * 1000, 40, 1.31/100, 3.2612], #in 2023 currency year
    'biomass': [2337.6116 * 126.38/100 * 1000, 30, 4.5269/100, 0] #OG in 2015 currency year, converted to 2023 currency using HICP
    }

In [ ]:
# Investment cost, lifetime, FOM and VOM per technology are from PyPSA technology-data
# (costs_2030.csv, see costs_dict above). Investment is annualized at a 7% discount rate
# (following Hofmann et al., Nature Energy 2025); FOM is a fraction of investment per year.
thermal['Inv_Cost_per_MWyr'] = thermal['Technology'].map(
    lambda x: annualize(costs_dict[x][0], r=0.07, lifetime=costs_dict[x][1])
)

thermal['Fixed_OM_Cost_per_MWyr'] = thermal['Technology'].map(lambda x: costs_dict[x][0] * costs_dict[x][2])
thermal['Var_OM_Cost_per_MWh'] = thermal['Technology'].map(lambda x: costs_dict[x][3])

In [ ]:
#efficiency of generators in pypsa in MWhelec/MWhfuel
#GenX is in MMBTU per MWh 
#1MWh = 3.412 MMBTU

thermal['Heat_Rate_MMBTU_per_MWh'] = 1/thermal['efficiency'] * 3.412

In [ ]:
thermal = thermal[[col for col in thermal_x.columns if col in thermal.columns]]
thermal_x = pd.concat([thermal_x, thermal], ignore_index=True)

In [ ]:
thermal_x['Existing_Cap_MW'] = 0 #Greenfield
thermal_x['Min_Cap_MW'] = 0 #Greenfield
thermal_x['New_Build'] = 1 #Greenfield
thermal_x['Cap_Size'] = thermal_x['Existing_Cap_MW']
thermal_x['Model'] = 2  # Simplified economic dispatch (no unit commitment)
thermal_x['Can_Retire'] = 1
thermal_x['cluster'] = 1
thermal_x['Max_Cap_MW'] = thermal_x['Max_Cap_MW'].replace(np.inf, -1)
thermal_x['Biomass'] = thermal_x['Resource'].str.endswith('biomass').astype(int)

# Fill missing values with 0's
thermal_x['Ramp_Up_Percentage'] = thermal_x['Ramp_Up_Percentage'].fillna(1)
thermal_x['Ramp_Dn_Percentage'] = thermal_x['Ramp_Dn_Percentage'].fillna(1)
thermal_x['Min_Power'] = thermal_x['Min_Power'].fillna(0)
thermal_x['Reg_Max'] = thermal_x['Reg_Max'].fillna(0)
thermal_x['Rsv_Max'] = thermal_x['Rsv_Max'].fillna(0)
thermal_x['Reg_Cost'] = thermal_x['Reg_Cost'].fillna(0)
thermal_x['Rsv_Cost'] = thermal_x['Rsv_Cost'].fillna(0)
thermal_x['Start_Fuel_MMBTU_per_MW'] = 0

In [ ]:
thermal_x.to_csv(output_folder + 'Thermal.csv', index = False)

## Fuels data

In [ ]:
#gas price = 40.88 eur/mwh, "annual average for THE in the European Gas Spot Index for year 2023" source: "Bundesnetzagentur 2024 market monitoring report"
#coal price = 150 usd / ton source: roughly from chart of https://www.iea.org/reports/coal-market-update-july-2023/prices
#lignite price = 1.5 eur/mwh_th; source: Agora Energiewende (Table 7-1) for time horizon 2025; https://www.agora-energiewende.de/fileadmin/Projekte/2021/2021_06_DE_Deutsche_Braunkohlewirtschaft/A-EW_248_Deutsche-Braunkohlenwirtschaft-2021_WEB.pdf
#biomass price = 26 eur/mwh; source: Vattenfall for 2020; https://group.vattenfall.com/siteassets/corporate/news_and_media/doc/biomass_report_-_final_-_compressed.pdf_13413687.pdf
#exchange rate 2023: 1 USD = 0.9241 source: https://www.exchangerates.org.uk/USD-EUR-spot-exchange-rates-history-2023.html
# coal energy content : 8.141 MWh / ton
# 1 MWh = 3.412 MMBtu

fuel_mapping = {
    'Gas':     40.88 / 3.412,                           # eur/mwh → eur/mmbtu
    'Biomass': 26    / 3.412,                           # eur/mwh → eur/mmbtu
    # 'Coal':    150 * 0.9241 / (8.141 * 3.412),        # usd/t → eur/mmbtu (uncomment if coal added back)
    # 'Lignite': 1.5 / 3.412,                           # eur/mwh → eur/mmbtu (uncomment if lignite added back)
}

#source: German environmental agency for year 2023, https://www.umweltbundesamt.de/system/files/medien/11850/publikationen/23_2024_cc_strommix_11_2024.pdf
#units of German environmental agency is tCO2/mwh_th, units of GenX in tCO2/MMBTU
emissions_mapping = {
    'Gas':     0.202 / 3.412,
    'Biomass': 0,
    # 'Coal':    0.338 / 3.412,   # (uncomment if coal added back)
    # 'Lignite': 0.407 / 3.412,   # (uncomment if lignite added back)
}

In [ ]:
fuel_types = list(fuel_mapping.keys())
prices_x = pd.DataFrame([], columns=fuel_types)
emissions_row = pd.DataFrame([], columns=fuel_types)

In [ ]:
for fuel, val in fuel_mapping.items():
    prices_x.loc[0, fuel] = val

for fuel, val in emissions_mapping.items():
    emissions_row.loc[0, fuel] = val

In [ ]:
t = 8736
emissions_row["Time_Index"] = 0

prices_x_t = pd.concat([prices_x] * (num_hours), ignore_index=True)
prices_x_t["Time_Index"] = np.arange(1, num_hours + 1)

In [ ]:
fuel_prices = pd.concat([emissions_row, prices_x_t], ignore_index=True)
fuel_prices = fuel_prices[["Time_Index"] + list(prices_x.columns)]

fuel_prices.to_csv(output_folder + 'Fuels_data.csv', index=False)

## VRE

In [ ]:
vre_x = pd.DataFrame([], columns = ['Resource',
'Zone',
'Num_VRE_Bins',
'New_Build',
'Can_Retire',
'Existing_Cap_MW',
'Max_Cap_MW',
'Min_Cap_MW',
'Inv_Cost_per_MWyr',
'Fixed_OM_Cost_per_MWyr',
'Var_OM_Cost_per_MWh',
'Fuel',
'Reg_Max',
'Rsv_Max',
'Reg_Cost',
'Rsv_Cost',
'region',
'cluster'])

In [ ]:
pyp_gen_vre_dict = {
    'Generator' : 'Resource',
    'bus' : 'Zone',
    # 'p_nom' : 'Existing_Cap_MW', # not used: greenfield
    'p_nom_max' : 'Max_Cap_MW',
    # 'p_nom_min' : 'Min_Cap_MW', # not used: greenfield
    'capital_cost' : 'Inv_Cost_raw',
    'marginal_cost' : 'Var_OM_Cost_per_MWh',
    }

In [ ]:
sources = ('solar', 'solar-hsat', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float') #offwind-dc and offwind-float not included because they connect via HVDC lines; also their p_nom = 0
vre = generator_list[generator_list['Generator'].str.endswith(sources)]
vre = vre[list(pyp_gen_vre_dict.keys())]
vre = vre.rename(columns = pyp_gen_vre_dict)

In [ ]:
tokens = vre['Resource'].str.split(r'\s+')
vre['Zone'] = tokens.str[1].astype(int) + 1
vre['Technology'] = tokens.str[-1]
vre['Resource'] = 'DE' + vre['Zone'].astype(str) + '_' + vre['Technology']
vre['region'] = 'DE' + vre['Zone'].astype(str)

In [ ]:
#dictionary of {'technology type' : [Investment (EUR/MW), lifetime (years), FOM (%/year), VOM (EUR/MWh)]}
#source of data: costs_2030.csv from pypsa
#to convert to 2023 currency year, HICP annual data from Eurostat were taken for EU-27:
#https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_aind/default/table?lang=en
#HICP_2015 = 100, HICP 2023 = 126.38, HICP 2020 =  105.76

costs_dict = {
    'onwind': [1095.8533 * 1000 * 126.38/100 , 30, 1.2167/100, 1.4286 * 126.38/100], #OG in 2015 currency year, converted to 2023 currency using HICP
    'solar': [383.7312 * 1000 * 126.38/105.76, 40, 2.4757/100, 0], #OG in 2020 currency year
    'solar-hsat': [454.4703 * 1000 * 126.38/105.76, 40,  2.2884/100, 0], #OG in 2020 currency year
    'offwind-ac': [1682.1226 * 1000 * 126.28/105.76 , 30, 2.3185/100, 0.0212 * 126.38/100], #VOM in 2015 currency year, investment in 2020 currency year
    'offwind-dc': [1682.1226 * 1000 * 126.28/105.76 , 30, 2.3185/100, 0.0212 * 126.38/100], #VOM in 2015 currency year, investment in 2020 currency year
    'offwind-float': [2350 * 1000 * 126.38/105.76, 20, 1.15/100, 0], #investment in 2020 currency; no VOM listed, we will override this with default 0.015 from config.yaml
    'ror': [3412.2266* 1000* 126.38/100, 80, 2/100, 0]} #OG in 2015 currency year

In [ ]:
vre['Inv_Cost_per_MWyr'] = vre['Technology'].map(
    lambda x: annualize(costs_dict[x][0], r=0.07, lifetime=costs_dict[x][1])
)

vre['Fixed_OM_Cost_per_MWyr'] = vre['Technology'].map(lambda x: costs_dict[x][0] * costs_dict[x][2])
vre['Var_OM_Cost_per_MWh'] = vre['Technology'].map(lambda x: costs_dict[x][3])

In [ ]:
#calculate investment cost for offshore technologies from pypsa capital cost b/c connection costs (dependent on distance) need to be added to offshore
#"The grid connection cost of offshore wind generators are added to the capital costs of the generator." source: https://pypsa-eur.readthedocs.io/en/v0.13.0-docs-fix/simplification.html
capital_cost_offshore = generator_list[['capital_cost', 'Generator']]
tokens = capital_cost_offshore['Generator'].str.split(r'\s+')
capital_cost_offshore['Zone'] = tokens.str[1].astype(int) + 1
capital_cost_offshore['Technology'] = tokens.str[-1]
capital_cost_offshore['Resource'] = 'DE' + capital_cost_offshore['Zone'].astype(str) + '_' + capital_cost_offshore['Technology']
capital_cost_offshore.set_index('Resource', inplace = True)
capital_cost_offshore = capital_cost_offshore['capital_cost']

offshore_techs = ['offwind-ac', 'offwind-dc', 'offwind-float']
offshore_mask = vre['Technology'].isin(offshore_techs)

#no VOM listed for offwind-float; 0.015 is the default in pypsa-eur config https://github.com/PyPSA/pypsa-eur-sec/blob/master/config.default.yaml
vre.loc[vre['Technology'] == 'offwind-float', 'Var_OM_Cost_per_MWh'] = 0.015

#pypsa formula: costs["capital_cost"] = (annuity + costs["FOM"] / 100) * costs["investment"]
#source: https://github.com/PyPSA/PyPSA/blob/v0.33.1/examples/notebooks/capacity-expansion-planning-single-node.ipynb
#because we also use a discount rate of 7%, we can do a simple subtraction
vre.loc[offshore_mask, 'Inv_Cost_per_MWyr'] = (vre.loc[offshore_mask, 'Resource'].map(capital_cost_offshore) - 
                                               vre.loc[offshore_mask, 'Fixed_OM_Cost_per_MWyr'])

In [ ]:
vre['Num_VRE_Bins'] = 1
vre['Existing_Cap_MW'] = 0 #Greenfield
vre['Min_Cap_MW'] = 0 #Greenfield
vre['New_Build'] = 1 #Greenfield
vre['Can_Retire'] = 1
vre[['Reg_Max', 'Rsv_Max', 'Reg_Cost', 'Rsv_Cost']] = 0
vre['Max_Cap_MW'] = vre['Max_Cap_MW'].replace(np.inf, -1)
vre['cluster'] = 1
vre['Fuel'] = 'None'

In [ ]:
vre = vre[[col for col in vre_x.columns if col in vre.columns]]
vre_x = pd.concat([vre_x, vre], ignore_index=True)

In [ ]:
vre_x.to_csv(output_folder + 'Vre.csv', index = False)

## Storage

In [ ]:
storage_pypsa = pd.read_csv(folder_path + f"DE_{n}_storage.csv")

In [ ]:
storage_x = pd.DataFrame([], columns = ['Resource',
'Zone',
'Model',
'New_Build',
'Can_Retire',
'Existing_Cap_MW',
'Existing_Cap_MWh',
'Max_Cap_MW',
'Max_Cap_MWh',
'Min_Cap_MW',
'Min_Cap_MWh',
'Inv_Cost_per_MWyr',
'Inv_Cost_per_MWhyr',
'Fixed_OM_Cost_per_MWyr',
'Fixed_OM_Cost_per_MWhyr',
'Var_OM_Cost_per_MWh',
'Var_OM_Cost_per_MWh_In',
'Self_Disch',
'Eff_Up',
'Eff_Down',
'Min_Duration',
'Max_Duration',
'Reg_Max',
'Rsv_Max',
'Reg_Cost',
'Rsv_Cost',
'region',
'cluster',
'LDS'])

In [ ]:
pyp_storage_dict = {
    'StorageUnit': 'Resource',
    'bus': 'Zone',
    'p_nom': 'Existing_Cap_MW',
    'p_nom_min': 'Min_Cap_MW',
    'p_nom_max': 'Max_Cap_MW',
    # 'marginal_cost': 'Var_OM_Cost_per_MWh',
    # 'capital_cost': 'Capital_Cost_per_MWyr',
    'max_hours': 'Max_Duration',
    'efficiency_store': 'Eff_Up',
    'efficiency_dispatch': 'Eff_Down',
    # 'p_nom_extendable': 'New_Build',
    'standing_loss': 'Self_Disch'
}

In [ ]:
storage_pypsa = storage_pypsa[storage_pypsa['StorageUnit'].str.endswith(store_sources)]
storage = storage_pypsa[list(pyp_storage_dict.keys())].rename(columns=pyp_storage_dict)

In [ ]:
tokens = storage['Resource'].str.split(r'\s+')
storage['Zone'] = tokens.str[1].astype(int) + 1
storage['Technology'] = tokens.str[-1]
storage['Resource'] = 'DE' + storage['Zone'].astype(str) + '_' + storage['Technology']
storage['region'] = 'DE' + storage['Zone'].astype(str)

In [ ]:
#Source: Danish Energy Agency Technology Data for Energy Storage (2025 projected costs, 2020 EUR)
#link: https://ens.dk/en/analyses-and-statistics/technology-data-energy-storage
#  Power component:              0.09  MEUR/MW   (inverter, power electronics)
#  Energy component:             0.10  MEUR/MWh  (battery cells)
#  Other project costs:          0.15  MEUR/MWh  (balance of plant, installation)
#  Total energy capex:           0.25  MEUR/MWh  (= 0.10 + 0.15)
#  Specific investment:          0.29  MEUR/MWh  (reference 3MWh/1.5MW system; = 0.09/2 + 0.25)
#  Fixed O&M (datasheet):        8.4   kEUR/MWh/yr  (Note N: 2.9% of specific investment;
#                                                     datasheet label says /MW but unit is /MWh)
#                                verified: 2.855% x 290,000 = 8,280 ~ 8,400 EUR/MWh/yr ✓
#  Technical lifetime:           15    years

#to convert to 2023 currency year, HICP annual data from Eurostat were taken for EU-27:
#https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_aind/default/table?lang=en
#HICP_2015 = 100, HICP 2023 = 126.38, HICP 2020 = 105.76
hicp = 126.38 / 105.76

power_capex_2020 = 0.09e6          # EUR/MW  (2020 EUR)
energy_capex_2020 = 0.25e6         # EUR/MWh (2020 EUR, = 0.10 + 0.15)
specific_investment_2020 = 0.29e6  # EUR/MWh (2020 EUR, from datasheet)

storage['Inv_Cost_per_MWyr'] = annualize(power_capex_2020 * hicp, r=0.07, lifetime=15)
storage['Inv_Cost_per_MWhyr'] = annualize(energy_capex_2020 * hicp, r=0.07, lifetime=15)

# Fixed O&M from DEA costs_battery report Table 16:
# FOM [EUR/MWh/yr] = (1.43% * P/E + 2.14%) * specific_investment [EUR/MWh]
# Expanding FOM_total [EUR/yr]:
#   = 1.43% * spec_inv * (P/E) * E  +  2.14% * spec_inv * E
#   = 1.43% * spec_inv * P          +  2.14% * spec_inv * E
storage['Fixed_OM_Cost_per_MWyr'] = 0.0143 * specific_investment_2020 * hicp
storage['Fixed_OM_Cost_per_MWhyr'] = 0.0214 * specific_investment_2020 * hicp
storage['Var_OM_Cost_per_MWh'] = 0
storage['Var_OM_Cost_per_MWh_In'] = 0

In [ ]:
# Fill remaining columns with defaults from the GenX example systems
storage['Model'] = 1
storage['New_Build'] = 1
storage['Can_Retire'] = 0
storage['Existing_Cap_MWh'] = 0
storage['Min_Cap_MWh'] = 0
storage['Var_OM_Cost_per_MWh'] = storage['Var_OM_Cost_per_MWh']/storage['Max_Duration']
storage['Var_OM_Cost_per_MWh_In'] = 0
storage['Min_Duration'] = 1
storage['Reg_Max'] = 0
storage['Rsv_Max'] = 0
storage['Reg_Cost'] = 0
storage['Rsv_Cost'] = 0
storage['cluster'] = 1
storage['Max_Cap_MW'] = storage['Max_Cap_MW'].replace(np.inf, -1)
storage['Max_Cap_MWh'] = storage['Max_Cap_MW']
storage['Min_Cap_MW'] = storage['Min_Cap_MW'].fillna(0)
storage['LDS'] = 1

In [ ]:
storage = storage[[col for col in storage_x.columns if col in storage.columns]]
storage_x = pd.concat([storage_x, storage], ignore_index=True)

In [ ]:
storage_x.to_csv(output_folder + 'Storage.csv', index=False)